# Forecasting WAPE and R2 Validation/Test Figures

Standalone presentation figures using saved forecasting evaluation metrics.

This notebook creates one validation-only figure and one test-only figure, each showing WAPE and R2. It does not retrain models or modify official artifacts.


In [ ]:

from pathlib import Path
import json

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUT_DIR = ROOT / 'notebooks_experiments'
FINAL_DIR = ROOT / 'backend' / 'Ai models' / 'data' / 'final' / 'Module 2'
NOTEBOOK_DIR = ROOT / 'backend' / 'Ai models' / 'notebooks' / 'Module 2'
REPORT_PDF = ROOT / 'rapport (1).pdf'

SOURCES = {
    'statistical': FINAL_DIR / 'statistical_comparison_weekly.json',
    'ml': FINAL_DIR / 'ml_comparison_summary.json',
    'lstm': FINAL_DIR / 'lstm_summary_weekly.json',
    'tft_notebook': NOTEBOOK_DIR / '06b_tft_weekly_aligned.ipynb',
    'report': REPORT_PDF,
}

for label, path in SOURCES.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing required source for {label}: {path}')

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

statistical = load_json(SOURCES['statistical'])
ml = load_json(SOURCES['ml'])
lstm = load_json(SOURCES['lstm'])

records = []

def add_model(model, family, source_path, key, summary):
    overall = summary['overall_metrics'][key]
    records.append({
        'model': model,
        'family': family,
        'source_file': str(source_path.relative_to(ROOT)).replace('\\', '/'),
        'val_wape': float(overall['val']['wape']),
        'val_r2': float(overall['val']['r2']),
        'test_wape': float(overall['test']['wape']),
        'test_r2': float(overall['test']['r2']),
    })

add_model('Prophet', 'Statistical', SOURCES['statistical'], 'prophet', statistical)
add_model('SARIMA', 'Statistical', SOURCES['statistical'], 'sarima', statistical)
add_model('XGBoost', 'Machine Learning', SOURCES['ml'], 'xgboost', ml)
add_model('LightGBM', 'Machine Learning', SOURCES['ml'], 'lightgbm', ml)
add_model('LSTM', 'Deep Learning', SOURCES['lstm'], 'lstm', {'overall_metrics': {'lstm': lstm['overall_metrics']}})

records.append({
    'model': 'TFT',
    'family': 'Deep Learning',
    'source_file': 'rapport (1).pdf Tables 6.10-6.11; backend/Ai models/notebooks/Module 2/06b_tft_weekly_aligned.ipynb output',
    'val_wape': 0.3471,
    'val_r2': 0.5475,
    'test_wape': 0.2739,
    'test_r2': 0.6362,
})

metrics = pd.DataFrame(records)

REPORT_EXPECTED = {
    'Prophet': {'val_wape': 0.3734, 'val_r2': 0.4937, 'test_wape': 0.2837, 'test_r2': 0.6196},
    'SARIMA': {'val_wape': 0.4019, 'val_r2': 0.4211, 'test_wape': 0.2867, 'test_r2': 0.5821},
    'XGBoost': {'val_wape': 0.3073, 'val_r2': 0.6579, 'test_wape': 0.2614, 'test_r2': 0.6707},
    'LightGBM': {'val_wape': 0.3061, 'val_r2': 0.6614, 'test_wape': 0.2617, 'test_r2': 0.6681},
    'LSTM': {'val_wape': 0.3175, 'val_r2': 0.6165, 'test_wape': 0.2676, 'test_r2': 0.6401},
    'TFT': {'val_wape': 0.3471, 'val_r2': 0.5475, 'test_wape': 0.2739, 'test_r2': 0.6362},
}

for _, row in metrics.iterrows():
    expected = REPORT_EXPECTED[row['model']]
    for col, expected_value in expected.items():
        actual = round(float(row[col]), 4)
        if actual != expected_value:
            raise AssertionError(f"{row['model']} {col} mismatch: figure metric={actual}, report={expected_value}")

metrics.to_csv(OUT_DIR / 'forecasting_wape_r2_validation_test_metrics_used.csv', index=False)

print('Data integrity checks')
print('- Validation plot: validation WAPE + validation R2 only.')
print('- Test plot: test WAPE + test R2 only.')
print('- Values match report Tables 6.8-6.11 at displayed precision.')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 11,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
})


def plot_split(split, order_by, ascending, ylabel, filename, highres_filename):
    plot_df = metrics.sort_values(order_by, ascending=ascending).reset_index(drop=True)
    x = np.arange(len(plot_df))
    width = 0.36

    fig, ax = plt.subplots(figsize=(12, 6.75), facecolor='white')
    ax.set_facecolor('white')

    bars_wape = ax.bar(
        x - width / 2,
        plot_df[f'{split}_wape'],
        width,
        label='WAPE',
        color='#4C78A8',
        edgecolor='black',
        linewidth=0.6,
    )
    bars_r2 = ax.bar(
        x + width / 2,
        plot_df[f'{split}_r2'],
        width,
        label='R2',
        color='#F58518',
        edgecolor='black',
        linewidth=0.6,
    )

    for bars in (bars_wape, bars_r2):
        ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=8, rotation=90)

    ax.set_ylabel(ylabel)
    ax.set_xlabel('Model')
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['model'])
    ax.set_ylim(0, 0.75)
    ax.legend(loc='upper right', frameon=True)
    ax.grid(axis='y', color='#d9d9d9', linewidth=0.8)
    ax.grid(axis='x', visible=False)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    fig.tight_layout(rect=[0.04, 0.04, 0.98, 0.98])

    png_path = OUT_DIR / filename
    highres_path = OUT_DIR / highres_filename
    fig.savefig(png_path, dpi=180, bbox_inches='tight', facecolor='white')
    fig.savefig(highres_path, dpi=320, bbox_inches='tight', facecolor='white')
    plt.close(fig)

    print(f'Wrote {png_path.relative_to(ROOT)}')
    print(f'Wrote {highres_path.relative_to(ROOT)}')
    print(plot_df[['model', f'{split}_wape', f'{split}_r2']].to_string(index=False, formatters={f'{split}_wape': '{:.4f}'.format, f'{split}_r2': '{:.4f}'.format}))

plot_split(
    split='val',
    order_by='val_wape',
    ascending=True,
    ylabel='Validation metric value',
    filename='figure_forecasting_validation_wape_r2.png',
    highres_filename='figure_forecasting_validation_wape_r2_highres.png',
)

plot_split(
    split='test',
    order_by='test_wape',
    ascending=True,
    ylabel='Test metric value',
    filename='figure_forecasting_test_wape_r2.png',
    highres_filename='figure_forecasting_test_wape_r2_highres.png',
)
